In [1]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="qwen3:8b"
)
llm

d:\MindsprintCourse\GenAI_Apps\chatbotWithMessageHistory\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatOllama(model='qwen3:8b')

In [3]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen3-embedding"
)

In [4]:
import sys
print(sys.executable)

d:\MindsprintCourse\GenAI_Apps\chatbotWithMessageHistory\venv\python.exe


In [5]:
import langchain
print(langchain.__version__)

1.2.13


In [8]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain



In [9]:
# load  the data
import bs4
loader = WebBaseLoader(
    web_path="https://lilianweng.github.io/posts/2023-06-23-agent/",
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)

In [11]:
docs = loader.load()

In [16]:
type(docs)

list

In [20]:
docs[0].page_content[:500]

'\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn'

In [23]:
# split the data into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorsotre = Chroma.from_documents(splits,embeddings)

In [ ]:
vectorsotre._collection.count() # 10 documents in the collection

63

In [25]:
retriever = vectorsotre.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000016BCE9E9A50>, search_kwargs={})

In [26]:
## prompt template

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [27]:
question_answer_chain = create_stuff_documents_chain(llm,prompt)

In [ ]:
rag_chain = create_retrieval_chain(retriever,question_answer_chain) 

In [34]:
response = rag_chain.invoke(
    {
        "input":"What is Self-Reflection?"
    }
)

In [37]:
type(response)

dict

In [38]:
response

{'input': 'What is Self-Reflection?',
 'context': [Document(id='1b2b9695-f1c6-4b8e-a9e6-e95d0aa288d5', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Self-Reflection#\nSelf-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.\nReAct (Yao et al. 2023) integrates reasoning and acting within LLM by extending the action space to be a combination of task-specific discrete actions and the language space. The former enables LLM to interact with the environment (e.g. use Wikipedia search API), while the latter prompting LLM to generate reasoning traces in natural language.\nThe ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:\nThought: ...\nAction: ...\nObservation: ...\n... (Repeated many times)'),
  Document(id='02393ace-0ed6-41ea-9

In [40]:
# from resposne docs i want to print answer
response["answer"]

'Self-Reflection is a process enabling autonomous agents to iteratively improve by refining past decisions and correcting mistakes, crucial in real-world tasks involving trial and error. It uses two-shot examples of failed trajectories and ideal reflections to guide future planning, storing up to three reflections in the agent’s working memory. This helps identify inefficiencies or hallucinations, such as redundant actions, to optimize decision-making.'

In [41]:
# if i ans another question we ge different output not related to self-reflection because we are not maintaining the conversation history in the chain. We can maintain the conversation history by using ConversationBufferMemory and passing it to the chain.
response = rag_chain.invoke(
    {
        "input":"How do we achieve it?"
    }
)

In [ ]:
'''
## prompt template

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm,prompt)

rag_chain = create_retrieval_chain(retriever,question_answer_chain) 

response = rag_chain.invoke(
    {
        "input":"What is Self-Reflection?"
    }
)

--- Important ---

## what is happening in this 
 When the user asks a question and `rag_chain.invoke()` is called, the retrieval chain starts executing. First, the retriever searches the vector store and fetches the most relevant document chunks based on the user's query using similarity search. Then those retrieved documents are passed to `question_answer_chain`.

`question_answer_chain` was created using `create_stuff_documents_chain(llm, prompt)`, whose job is to combine or "stuff" all retrieved documents into the `{context}` placeholder of the prompt. At the same time, the user question is inserted into `{input}`.

Finally, this complete prompt is sent to the LLM, and the LLM generates the final answer based on the retrieved context.

'''

In [42]:
response

{'input': 'How do we achieve it?',
 'context': [Document(id='02393ace-0ed6-41ea-9a47-937f310730dc', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Conversatin samples:\n[\n  {\n    "role": "system",'),
  Document(id='7d74479d-637d-462c-ade5-e652312c1175', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='GOALS:\n\n1. {{user-provided goal 1}}\n2. {{user-provided goal 2}}\n3. ...\n4. ...\n5. ...\n\nConstraints:\n1. ~4000 word limit for short term memory. Your short term memory is short, so immediately save important information to files.\n2. If you are unsure how you previously did something or want to recall past events, thinking about similar events will help you remember.\n3. No user assistance\n4. Exclusively use the commands listed in double quotes e.g. "command name"\n5. Use subprocesses for commands that will not terminate within a few minutes'),
  Document(id='3b6c6d64-3d87-485e-bed2-d28cc7b12d10

# Adding chat history


In [ ]:
from langchain_classic.chains import create_history_aware_retriever ## in this retriver will aware of the conversation history and will use it to retrieve the relevant documents from the vectorstore. 
'''
Yeh kya hai?

Yeh LangChain ka ek helper function hai jo ek special retriever chain banata hai.

Iska kaam:

Agar user ka latest question chat history par dependent ho, toh:

pehle us question ko rewrite/reformulate kare
phir rewritten question se documents retrieve kare
'''

from langchain_core.prompts import MessagesPlaceholder
# this messagePlaceholder will be used to store the conversation history and will be passed to the retriever to retrieve the relevant documents from the vectorstore.


contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)


In [ ]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt)
'''
Yeh kya bana raha hai?

Ek aisa retriever jo directly raw user question use nahi karega.
Pehle wo dekhega:

kya chat history hai?
kya latest question ambiguous hai?
agar haan, toh LLM se usse rewrite karvayega
phir retriever ko rewritten question dega

'''

In [46]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000016BCE9E9A50>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChun

In [48]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [49]:
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)

In [50]:
rag_chain = create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [52]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="What is Self-Reflection"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2="Tell me more about it?"
response2=rag_chain.invoke({"input":question,"chat_history":chat_history})
print(response2['answer'])

Self-Reflection is a process enabling autonomous agents to improve by analyzing past actions, correcting mistakes, and optimizing future decisions. It integrates reasoning (via frameworks like ReAct) and acting, using language and discrete actions to interact with environments. The Reflexion framework detects inefficient planning or hallucination (repetitive actions without progress) and uses reflective examples to guide better strategies.


## now do it with session management

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

chat_history = []

question = "What is Self-Reflection"
response1 = rag_chain.invoke({"input": question, "chat_history": chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2 = "Tell me more about it?"
response2 = rag_chain.invoke({"input": question2, "chat_history": chat_history})

print(response2["answer"])




'''
1) Jab user first question ask karta hai, for example "What is Self-Reflection", tab `rag_chain.invoke()` execute hota hai. Isme user ka input aur `chat_history` pass hota hai. Initially `chat_history` empty hoti hai.

2) `rag_chain` already `create_retrieval_chain()` se bana hua hota hai. Jab yeh run hota hai, sabse pehle `history_aware_retriever` execute hota hai. Iska kaam hota hai user ke latest question ko chat history ke context me samajhna. Agar question ambiguous ya follow-up type ka ho, toh yeh LLM ki help se us question ko standalone question me rewrite karta hai. Agar question already clear hai, toh same question use hota hai.

3) Uske baad actual `retriever` rewritten ya original question ke basis par vector store se relevant documents/chunks retrieve karta hai.

4) Phir yeh retrieved documents `question_answer_chain` ko pass kiye jaate hain. `question_answer_chain`, jo `create_stuff_documents_chain()` se bana hota hai, retrieved docs ko combine karta hai aur unhe prompt ke `{context}` placeholder me insert karta hai. Saath hi user ka question `{input}` me aur chat history `MessagesPlaceholder("chat_history")` ke through prompt me inject hoti hai.

5) Ab final prompt LLM ko diya jata hai. LLM retrieved context + current question + chat history ke basis par final answer generate karta hai.

6) First answer milne ke baad hum user ka question aur AI ka response manually `chat_history` list me store kar dete hain using `HumanMessage` and `AIMessage`.

7) Jab user second follow-up question ask karta hai, for example "Tell me more about it?", tab dobara same `rag_chain.invoke()` run hota hai. Is baar `chat_history` empty nahi hoti, isliye `history_aware_retriever` samajh pata hai ki "it" kis cheez ko refer kar raha hai, aur usse standalone question me convert karke better retrieval karta hai.

8) Fir retriever relevant docs laata hai aur LLM us context aur history ke basis par more relevant conversational answer generate karta hai.

'''